# 📘 Notebook 03: Model Training, Hierarchical Reconciliation & Production Inference
**Project:** Retail Multi-Store Hierarchical Demand Forecaster (Kaggle Walmart M5 Dataset)

---
### Objectives:
1. Load processed feature panel data and configure Time-Series Cross Validation (rolling-window splits with no lookahead bias).
2. Benchmark baseline models (Naive, Seasonal Naive) against **LightGBM with Tweedie Loss**.
3. Track experiments, parameters, and loss metrics using **MLflow**.
4. Apply **Hierarchical Time Series (HTS) Reconciliation** (MinT / Bottom-Up) to guarantee consistent forecasts from SKU to State and National levels.
5. Evaluate accuracy metrics (**WAPE**, **RMSSE**, **RMSE**).
6. Execute the **Production Inventory Reorder Engine** (Safety Stock & Reorder Point alerts).

In [ ]:
# 1. Environment Setup & Data Loading
import sys
import os
import pandas as pd

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.utils import set_seed
set_seed(42)

In [ ]:
# 2. Train LightGBM Model with Tweedie Loss & MLflow Tracking
from src.train import train_lightgbm_model

model, val_preds, feature_importance = train_lightgbm_model(
    processed_data_dir=os.path.join(project_root, 'data', 'processed'),
    models_dir=os.path.join(project_root, 'models')
)

In [ ]:
# 3. Hierarchical Time Series (HTS) Reconciliation
from src.hts import reconcile_hierarchical_forecasts

reconciled_df = reconcile_hierarchical_forecasts(
    predictions_df=val_preds,
    models_dir=os.path.join(project_root, 'models')
)

In [ ]:
# 4. Model Evaluation & Metric Performance (WAPE, RMSSE, RMSE)
from src.evaluate import evaluate_forecast_accuracy

metrics_report = evaluate_forecast_accuracy(
    reconciled_df=reconciled_df,
    reports_dir=os.path.join(project_root, 'reports')
)

In [ ]:
# 5. Production Inventory Reorder Engine (Safety Stock & Reorder Point Alerts)
from src.inference import run_inventory_reorder_pipeline

reorder_table = run_inventory_reorder_pipeline(
    reconciled_df=reconciled_df,
    service_level=0.95,
    lead_time_days=7
)

print('\n✅ Notebook 03 Execution Complete! Complete End-to-End MLOps Pipeline Verified!')